# OptiCell — BBBC039 validation on Google Colab

**Measured metrics only.** Runs classical threshold and optional Cellpose on U2OS nuclei (BBBC039).

**Setup:** Runtime → Change runtime type → **T4 GPU** (or any GPU).

Repo: https://github.com/Virelion-Biotech/Virelion-OptiCell

## 1. Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU — enable GPU runtime for Cellpose')

## 2. Install dependencies

In [ ]:
!pip -q install opencv-python-headless tifffile scipy pandas packaging cellpose
print('deps ok')

## 3. Clone OptiCell

In [ ]:
import os
from pathlib import Path

WORK = Path('/content/Virelion-OptiCell')
if not (WORK / 'qc_pipeline.py').exists():
    !git clone --depth 1 https://github.com/Virelion-Biotech/Virelion-OptiCell.git /content/Virelion-OptiCell
else:
    print('repo already present')

%cd /content/Virelion-OptiCell
!pip -q install -e .
print('cwd:', os.getcwd())

## 4. Download BBBC039

In [ ]:
import urllib.request
import zipfile
from pathlib import Path

DATA = Path('data/bbbc039')
RAW = DATA / 'raw'
IMG_DIR = DATA / 'images'
MSK_DIR = DATA / 'masks'
RAW.mkdir(parents=True, exist_ok=True)

urls = {
    'images.zip': 'https://data.broadinstitute.org/bbbc/BBBC039/images.zip',
    'masks.zip': 'https://data.broadinstitute.org/bbbc/BBBC039/masks.zip',
}

for name, url in urls.items():
    dest = RAW / name
    if dest.exists() and dest.stat().st_size > 0:
        print('skip download', name)
        continue
    print('downloading', name, '...')
    urllib.request.urlretrieve(url, dest)
    print('  MB:', round(dest.stat().st_size / 1e6, 1))

for zname, out in [('images.zip', IMG_DIR), ('masks.zip', MSK_DIR)]:
    marker = out / '.extracted'
    if marker.exists():
        print('skip extract', zname)
        continue
    out.mkdir(parents=True, exist_ok=True)
    print('extracting', zname, '...')
    with zipfile.ZipFile(RAW / zname) as zf:
        zf.extractall(out)
    marker.write_text('ok')
print('BBBC039 ready')

## 5. Config

In [ ]:
MAX_IMAGES = 50
RUN_THRESHOLD = True
RUN_CELLPOSE = True
CELLPOSE_MODEL = 'cpsam'
USE_GPU = True

## 6. Run validation

In [ ]:
import subprocess
import sys

def run_backend(backend, gpu=False):
    cmd = [
        sys.executable,
        'scripts/run_bbbc039_validation.py',
        '--max-images', str(MAX_IMAGES),
        '--backend', backend,
        '--skip-download',
        '--out-dir', 'outputs/bbbc039_validation',
    ]
    if backend == 'cellpose':
        cmd += ['--cellpose-model', CELLPOSE_MODEL]
        if gpu:
            cmd.append('--gpu')
    print('>>>', ' '.join(cmd))
    r = subprocess.run(cmd)
    if r.returncode != 0:
        raise SystemExit(backend + ' failed with code ' + str(r.returncode))

if RUN_THRESHOLD:
    run_backend('threshold')

if RUN_CELLPOSE:
    run_backend('cellpose', gpu=USE_GPU)

## 7. Show measured summaries

In [ ]:
import json
from pathlib import Path

out = Path('outputs/bbbc039_validation')
files = sorted(out.glob('bbbc039_*_n*.json'))
if not files:
    print('No result JSON yet')
else:
    for f in files:
        data = json.loads(f.read_text())
        s = data['summary']
        print('=' * 60)
        print('File:', f.name)
        print('backend=', data.get('backend'), 'n=', data.get('n_scored'), 'gpu=', data.get('gpu'))
        iou = s.get('iou', s.get('pixel_iou_mean'))
        dice = s.get('dice', s.get('pixel_dice_mean'))
        f1 = s.get('f1', s.get('instance_f1_mean'))
        ac = s.get('absolute_count_error_mean', s.get('absolute_count_error'))
        rc = s.get('relative_count_error_mean', s.get('relative_count_error'))
        print('  IoU:  ', round(float(iou), 4))
        print('  Dice: ', round(float(dice), 4))
        print('  F1:   ', round(float(f1), 4))
        print('  |count err| mean:', round(float(ac), 2))
        print('  rel count err:   ', rc)
    print('=' * 60)
    print('These numbers are measured only — not fabricated.')

## 8. Download results

Or use the Files panel: `Virelion-OptiCell/outputs/bbbc039_validation/`

In [ ]:
from google.colab import files
from pathlib import Path

for f in sorted(Path('outputs/bbbc039_validation').glob('bbbc039_*')):
    print('download', f)
    files.download(str(f))